# 01 — Exploratory Data Analysis: Avocado Prices

This notebook performs an initial exploration of the **Hass Avocado Board** dataset (`avocado.csv`).

**Goals**
1. Load the raw data and inspect its structure.
2. Understand column types, missing values, and basic statistics.
3. Visualise price distributions, trends over time, and regional differences.
4. Document any data-quality issues to address in later preprocessing.

**Dataset columns**
| Column | Description |
|---|---|
| `Date` | Observation date (weekly) |
| `AveragePrice` | Average retail price of a single avocado |
| `Total Volume` | Total number of avocados sold |
| `4046` / `4225` / `4770` | Volume sold by PLU code (size) |
| `Total Bags` / `Small Bags` / `Large Bags` / `XLarge Bags` | Volume sold in bags |
| `type` | Conventional or organic |
| `year` | Year of observation |
| `region` | City or region of the observation |

## 1. Imports & Configuration

In [ ]:
# Core data handling
import pandas as pd
import numpy as np
from pathlib import Path

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Notebook display helpers
from IPython.display import display, Markdown

# ---------- Plot style ----------
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 5)

# ---------- Paths ----------
DATA_RAW = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")
DATA_OUTPUTS = Path("../data/outputs")

print("✔ Imports loaded successfully")

## 2. Load the Raw Data

Read `avocado.csv` and perform a quick sanity check: shape, first rows, and column types.

In [ ]:
# Load dataset — the first unnamed column is the original row index, so we drop it
df = pd.read_csv(DATA_RAW / "avocado.csv", index_col=0)

# Parse the Date column to datetime
df["Date"] = pd.to_datetime(df["Date"])

# Sort by date for consistent time-series operations
df = df.sort_values("Date").reset_index(drop=True)

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

## 3. Data Overview

Check column data types, memory usage, and missing values.

In [ ]:
df.info()

In [ ]:
# Missing values per column
missing = df.isnull().sum()
missing = missing[missing > 0]
if missing.empty:
    print("No missing values found.")
else:
    display(missing.to_frame("missing_count"))

In [ ]:
# Descriptive statistics for numeric columns
df.describe().T

In [ ]:
# Categorical columns — unique values
print(f"Types     : {df['type'].unique()}")
print(f"Years     : {sorted(df['year'].unique())}")
print(f"Regions   : {df['region'].nunique()} unique regions")
print(f"Date range: {df['Date'].min().date()} → {df['Date'].max().date()}")

## 4. Price Distribution

Visualise how `AveragePrice` is distributed overall and split by avocado type.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall distribution
sns.histplot(df["AveragePrice"], kde=True, bins=50, ax=axes[0])
axes[0].set_title("AveragePrice — Overall Distribution")
axes[0].set_xlabel("Price (USD)")

# By type
sns.boxplot(data=df, x="type", y="AveragePrice", ax=axes[1])
axes[1].set_title("AveragePrice by Type")
axes[1].set_xlabel("")
axes[1].set_ylabel("Price (USD)")

plt.tight_layout()
plt.show()

## 5. Price Trends Over Time

Plot weekly average price aggregated across all regions, split by type.

In [ ]:
# Weekly average price by type (aggregated across regions)
weekly_price = (
    df.groupby(["Date", "type"])["AveragePrice"]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(14, 5))
for avocado_type, group in weekly_price.groupby("type"):
    ax.plot(group["Date"], group["AveragePrice"], label=avocado_type, alpha=0.85)

ax.set_title("Weekly Average Avocado Price by Type")
ax.set_xlabel("Date")
ax.set_ylabel("Price (USD)")
ax.legend(title="Type")
plt.tight_layout()
plt.show()

## 6. Volume Analysis

Explore total volume sold and how it relates to price.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Volume distribution (log scale for readability)
sns.histplot(df["Total Volume"], bins=60, log_scale=True, ax=axes[0])
axes[0].set_title("Total Volume Distribution (log scale)")
axes[0].set_xlabel("Total Volume")

# Price vs Volume scatter
sns.scatterplot(
    data=df.sample(3000, random_state=42),
    x="Total Volume", y="AveragePrice",
    hue="type", alpha=0.5, ax=axes[1],
)
axes[1].set_xscale("log")
axes[1].set_title("Price vs. Volume (sampled)")
axes[1].set_xlabel("Total Volume (log)")
axes[1].set_ylabel("Price (USD)")

plt.tight_layout()
plt.show()

## 7. Regional Comparison

Compare average prices across the top 15 regions by volume.

In [ ]:
# Top 15 regions by total volume
top_regions = (
    df.groupby("region")["Total Volume"]
    .sum()
    .nlargest(15)
    .index
)

df_top = df[df["region"].isin(top_regions)]

fig, ax = plt.subplots(figsize=(14, 6))
sns.boxplot(
    data=df_top, x="region", y="AveragePrice",
    hue="type", order=top_regions, ax=ax,
)
ax.set_title("Average Price Distribution — Top 15 Regions by Volume")
ax.set_xlabel("")
ax.set_ylabel("Price (USD)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 8. Correlation Heatmap

Examine linear relationships between numeric features.

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation Matrix — Numeric Features")
plt.tight_layout()
plt.show()

## Next Steps

Key observations to carry into **02_feature_engineering.ipynb**:

- **Organic avocados** are consistently more expensive than conventional ones.
- Volume features are highly right-skewed — consider **log transforms**.
- Bag-related columns are strongly correlated with each other — may warrant **aggregation or feature selection**.
- The `region` column contains both cities and aggregate regions (e.g. `TotalUS`) — filter or encode carefully.
- The `year` column is redundant with `Date` — can be dropped after extracting richer time features.

---
*End of EDA notebook.*